
# Thí nghiệm 02 — Huấn luyện dự báo tốc độ gió ERA5

## 1. Thiết lập dữ liệu và môi trường

### 1.1. Mục tiêu

Notebook thực hiện huấn luyện và đánh giá bốn mô hình dự báo chuỗi thời gian trên dữ liệu tốc độ gió ERA5 tại 10 địa điểm thuộc nước Đức.

Các mô hình gồm:

1. LSTM.
2. PatchTST kết hợp toán tử Koopman.
3. Autoformer kết hợp toán tử Koopman.
4. Informer kết hợp toán tử Koopman.

Mục tiêu là so sánh sai số dự báo và khảo sát quá trình tối ưu của các mô hình trong cùng một cấu hình thực nghiệm.

### 1.2. Dữ liệu sử dụng

Dữ liệu được chuẩn bị trong:

`06_climate_era5_data.ipynb`

Tệp đầu vào:

`data/era5_wind_10sites.pt`

Ma trận tốc độ gió có kích thước:

$$
W\in\mathbb{R}^{400\times10}
$$

Trong đó:

- 400 là số thời điểm.
- 10 là số địa điểm.
- Đơn vị gốc là m/s.

Dữ liệu ERA5 có tần suất một giờ và được trích xuất từ 400 thời điểm đầu tiên của tháng 1/2020.

### 1.3. Cấu hình thực nghiệm

| Tham số | Giá trị |
|---|---|
| Input length | 120 |
| Forecast horizon | 30 |
| d_model | 48 |
| Epochs | 4000 |
| Learning rate | 0.0003 |
| Lyapunov weight | 0.1 |
| Random seed | 7 |
| Số mô hình | 4 |

Với 400 thời điểm, số cửa sổ được tạo là:

$$
400-120-30=250
$$

Kích thước input:

$$
X\in\mathbb{R}^{250\times120\times10}
$$

Kích thước target:

$$
Y\in\mathbb{R}^{250\times300}
$$

### 1.4. Môi trường huấn luyện

Lần thực nghiệm đã lưu được thực hiện trên Kaggle với GPU NVIDIA Tesla T4.

Notebook sử dụng PyTorch để huấn luyện và lưu các checkpoint.

Trong lần tái cấu trúc này, mã nguồn được tổ chức để có thể truy cập dữ liệu từ thư mục project trên máy hoặc từ môi trường Kaggle.

### 1.5. Tổ chức kết quả

Các kết quả được lưu tại:

`results/era5_wind_e4000/`

Thư mục chứa:

- Checkpoint cuối cùng của bốn mô hình.
- Tệp tổng hợp `metrics.csv`.
- Dữ liệu dự báo, nhãn thực tế và lịch sử huấn luyện nằm trong checkpoint.
- Các hình dự báo và phân tích kết quả.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler


# 1. Xác định môi trường thực thi và đường dẫn
kaggle_input = Path("/kaggle/input")

if kaggle_input.is_dir():
    # Môi trường Kaggle: dữ liệu được đính kèm qua Kaggle Datasets
    data_files = list(
        kaggle_input.rglob("era5_wind_10sites.pt")
    )

    model_files = list(
        kaggle_input.rglob("02_climate_model.ipynb")
    )

    if len(data_files) != 1:
        raise FileNotFoundError(
            f"Expected one ERA5 dataset, found: {data_files}"
        )

    if len(model_files) != 1:
        raise FileNotFoundError(
            f"Expected one model notebook, found: {model_files}"
        )

    data_path = data_files[0]
    model_notebook = model_files[0]

    experiment_dir = Path("/kaggle/working")
    results_dir = experiment_dir / "era5_wind_e4000"

else:
    # Môi trường local: tìm thư mục gốc của project
    project_root = next(
        (
            p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "experiments" / "02_climate").is_dir()
        ),
        None
    )

    if project_root is None:
        raise FileNotFoundError(
            "Không tìm thấy thư mục gốc của project."
        )

    experiment_dir = (
        project_root / "experiments" / "02_climate"
    )

    data_path = (
        experiment_dir / "data" / "era5_wind_10sites.pt"
    )

    model_notebook = (
        experiment_dir / "02_climate_model.ipynb"
    )

    results_dir = (
        experiment_dir / "results" / "era5_wind_e4000"
    )

    if not data_path.is_file():
        raise FileNotFoundError(data_path)

    if not model_notebook.is_file():
        raise FileNotFoundError(model_notebook)


# 2. Nạp dữ liệu ERA5 đã chuẩn bị
dataset = torch.load(
    data_path,
    map_location="cpu",
    weights_only=True
)

wind_raw = dataset["wind_raw"].numpy()
location_names = dataset["location_names"]


# 3. Cấu hình thực nghiệm
patch_len = 120
horizon = 30
d_model = 48

epochs = 4000
learning_rate = 3e-4
lyap_weight = 0.1
seed = 7

input_dim = wind_raw.shape[1]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# 4. Chuẩn hóa Min-Max
scaler = MinMaxScaler()
wind_scaled = scaler.fit_transform(wind_raw[:400])


# 5. Tạo cửa sổ trượt
X, Y = [], []

for i in range(len(wind_scaled) - patch_len - horizon):
    X.append(wind_scaled[i:i + patch_len])

    Y.append(
        wind_scaled[i + patch_len:i + patch_len + horizon]
        .T.reshape(-1)
    )


# 6. Chuyển dữ liệu sang tensor PyTorch
x_in = torch.tensor(
    np.stack(X),
    dtype=torch.float32
)

x_out = torch.tensor(
    np.stack(Y),
    dtype=torch.float32
)


# 7. Kiểm tra dữ liệu
assert wind_raw.shape == (400, 10)
assert np.isfinite(wind_raw).all()

assert x_in.shape == (250, 120, 10)
assert x_out.shape == (250, 300)

assert len(location_names) == 10


# 8. Tổng hợp thông tin thực nghiệm
print("=== ERA5 WIND SPEED ===")

print("Device:", device)
print("Dataset:", data_path)
print("Model notebook:", model_notebook)

print("Raw:", wind_raw.shape)
print("X:", x_in.shape)
print("Y:", x_out.shape)

print("Locations:", len(location_names))
print("Patch length:", patch_len)
print("Horizon:", horizon)
print("d_model:", d_model)

print("Epochs:", epochs)
print("Training runs:", 4)
print("Results:", results_dir)

=== KAGGLE ERA5 WIND SPEED ===
GPU: Tesla T4
Dataset: /kaggle/input/datasets/tunnghanguynng/era5wind/era5_wind_10sites.pt
Model notebook: /kaggle/input/datasets/tunnghanguynng/climate/02_climate_model.ipynb
Raw: (400, 10)
X: torch.Size([250, 120, 10])
Y: torch.Size([250, 300])
Locations: 10
Patch length: 120
Horizon: 30
d_model: 48
Epochs: 4000
Training runs: 4
Results: /kaggle/working/era5_wind_e4000



## 2. Nạp các kiến trúc dự báo

### 2.1. Mục tiêu

Các lớp mô hình được sử dụng từ notebook:

`02_climate_model.ipynb`

Thay vì sao chép định nghĩa kiến trúc, notebook đọc mã nguồn và trích xuất các lớp cần thiết.

Phương pháp này giúp giữ thống nhất kiến trúc giữa các nhánh thực nghiệm Climate.

### 2.2. Phương pháp nạp mô hình

Notebook Model được đọc ở định dạng JSON.

Sử dụng thư viện `ast` để trích xuất:

- Các câu lệnh import.
- Các định nghĩa hàm.
- Các định nghĩa lớp.

Những thành phần này được biên dịch và nạp vào môi trường thực thi.

Các câu lệnh đọc dữ liệu và các thao tác khác không thuộc nhóm định nghĩa được bỏ qua.

### 2.3. Các mô hình sử dụng

| Mô hình | Lớp |
|---|---|
| LSTM | `SimpleLSTMForecaster` |
| PatchTST | `Koopformer_PatchTST` |
| Autoformer | `Koopformer_Autoformer` |
| Informer | `Koopformer_Informer` |

Hàm `create_model()` nhận tên mô hình và trả về kiến trúc tương ứng với cấu hình ERA5.

### 2.4. Kích thước đầu vào và đầu ra

Số kênh:

$$
C=10
$$

Chân trời dự báo:

$$
H=30
$$

Đầu ra của tất cả mô hình có kích thước:

$$
B\times(HC)=B\times300
$$

Trong đó $B$ là số mẫu của batch.

LSTM sử dụng hidden size 48 và hai tầng.

Các mô hình Koopman sử dụng những backbone được định nghĩa trong notebook Model, với `d_model=48`.

### 2.5. Bảo toàn kiến trúc

Không thay đổi tên lớp, tên tham số, cấu hình tầng hoặc phương thức `forward()` của các mô hình.

Điều này nhằm bảo đảm các checkpoint ERA5 đã huấn luyện vẫn tương thích với kiến trúc.


In [ ]:
import ast
import json

# 1. Read model notebook from Kaggle Dataset
with open(model_notebook, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# 2. Extract imports, functions and classes only
allowed_nodes = (
    ast.Import,
    ast.ImportFrom,
    ast.ClassDef,
    ast.FunctionDef,
    ast.AsyncFunctionDef
)

definitions = []

for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    source = cell["source"]
    if isinstance(source, list):
        source = "".join(source)

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(node, allowed_nodes):
            definitions.append(node)

# 3. Execute model definitions
module = ast.Module(body=definitions, type_ignores=[])
ast.fix_missing_locations(module)

exec(
    compile(module, str(model_notebook), "exec"),
    globals()
)

# 4. Model factory
def create_model(model_name):
    if model_name == "LSTM":
        return SimpleLSTMForecaster(
            input_dim, 48, 2, horizon
        )

    if model_name == "PatchTST":
        return Koopformer_PatchTST(
            input_dim, patch_len, horizon,
            patch_len=patch_len, d_model=d_model
        )

    if model_name == "Autoformer":
        return Koopformer_Autoformer(
            input_dim, patch_len, horizon,
            patch_len=patch_len, d_model=d_model
        )

    if model_name == "Informer":
        return Koopformer_Informer(
            input_dim, patch_len, horizon,
            patch_len=patch_len, d_model=d_model
        )

    raise ValueError(f"Unknown model: {model_name}")


=== ERA5 MODEL VALIDATION ===
LSTM         Parameters: 45,036 | Output: (2, 300)
PatchTST     Parameters: 133,932 | Output: (2, 300)
Autoformer   Parameters: 96,726 | Output: (2, 300)
Informer     Parameters: 133,884 | Output: (2, 300)

All four models initialized successfully.



## 3. Hàm huấn luyện và tối ưu mô hình

### 3.1. Mục tiêu

Hàm `train_era5_model()` thực hiện tối ưu trọng số cho một mô hình dự báo ERA5 trong 4000 epoch.

Cả bốn mô hình sử dụng cùng quy trình huấn luyện, với sự khác biệt ở hệ số chính quy hóa Lyapunov.

### 3.2. Hàm mất mát dự báo

Sai số được tính bằng Mean Squared Error:

$$
\mathcal{L}_{\mathrm{MSE}}
=
\frac{1}{N}
\sum_{i=1}^{N}
(\hat y_i-y_i)^2
$$

Trong đó $\hat y_i$ là giá trị dự báo và $y_i$ là giá trị thực tế.

### 3.3. Chính quy hóa Lyapunov

Các biến thể Koopman sử dụng thành phần chính quy hóa nhằm hạn chế sự gia tăng năng lượng trong không gian tiềm ẩn.

Ma trận Lyapunov được chọn:

$$
P=I
$$

Hàm năng lượng:

$$
V(z)=z^\top Pz
$$

Thành phần phạt:

$$
\mathcal{L}_{\mathrm{Lyap}}
=
\frac{1}{B}
\sum_{i=1}^{B}
\max(0,V(z_i')-V(z_i))
$$

Trong đó $z_i$ và $z_i'$ lần lượt là biểu diễn trước và sau toán tử Koopman.

### 3.4. Hàm mục tiêu tổng hợp

$$
\mathcal{L}
=
\mathcal{L}_{\mathrm{MSE}}
+
\lambda\mathcal{L}_{\mathrm{Lyap}}
$$

Hệ số chính quy hóa:

- LSTM: $\lambda=0$.
- PatchTST, Autoformer, Informer kết hợp Koopman: $\lambda=0.1$.

### 3.5. Quy trình huấn luyện

Hàm sử dụng:

- Adam Optimizer.
- Learning rate `3e-4`.
- 4000 epoch.
- Full-batch training.
- Random seed bằng 7.

Mỗi epoch thực hiện truyền xuôi, tính hàm mất mát, lan truyền ngược và cập nhật trọng số.

Sau huấn luyện, mô hình được chuyển sang chế độ đánh giá và tạo dự báo trên chính các cửa sổ huấn luyện.

### 3.6. Giá trị trả về

Hàm trả về:

| Giá trị | Nội dung |
|---|---|
| `predictions` | Các dự báo sau huấn luyện |
| `losses` | Lịch sử hàm mất mát |
| `eigs` | Lịch sử chỉ số theo dõi tham số Koopman |

Trong code, chỉ số `eigs` được xác định bằng:

`max(abs(sigmoid(S_raw)))`

Đại lượng này không phải là trị riêng lớn nhất của ma trận Koopman và chưa nhân với hệ số giới hạn 0.99.

Vì vậy, khi phân tích biểu đồ ổn định, cần phân biệt chỉ số tham số hóa được ghi nhận với giá trị kỳ dị thực tế của toán tử.

### 3.7. Giới hạn đánh giá

Các dự báo sau huấn luyện được tính trên chính dữ liệu đã sử dụng để tối ưu mô hình.

Kết quả MSE và MAE của nhánh này là metric trên dữ liệu huấn luyện, không phải kết quả kiểm tra trên một tập Test độc lập.


In [ ]:
import torch.nn as nn
import torch.optim as optim

# 1. Thiết lập hạt giống ngẫu nhiên
def set_seed(seed=7):
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(seed)

# 2. Truy cập thuộc tính lồng nhau của mô hình
def _nested_getattr(obj, path):
    for p in path.split("."):
        obj = getattr(obj, p, None)
        if obj is None:
            return None
    return obj

# 3. Hàm huấn luyện
def train_era5_model(
    model, x_in, x_out,
    epochs=4000, lr=3e-4,
    koop_attr="koop", lyap_weight=0.1
):
    model.to(device)
    x_in = x_in.to(device)
    x_out = x_out.to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    mse = nn.MSELoss()

    losses, eigs = [], []

    # Khởi tạo ma trận Lyapunov P = I
    with torch.no_grad():
        try:
            _, z, z_next = model(
                x_in[:1], return_latents=True
            )
        except ValueError:
            z = z_next = torch.zeros(
                1, model.fc.in_features, device=device
            )

    P = torch.eye(z.shape[1], device=device)

    # Vòng lặp huấn luyện
    for ep in range(epochs):
        model.train()
        optimizer.zero_grad()

        pred, z, z_next = model(
            x_in, return_latents=True
        )

        pred_loss = mse(pred, x_out)

        if lyap_weight:
            zp = torch.einsum("bi,ij->bj", z, P)
            znp = torch.einsum("bi,ij->bj", z_next, P)

            lyap = torch.relu(
                (znp * z_next).sum(1)
                - (zp * z).sum(1)
            ).mean()
        else:
            lyap = torch.zeros(1, device=device)

        loss = pred_loss + lyap_weight * lyap

        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        # Theo dõi tham số giá trị kỳ dị của toán tử Koopman
        obj = (
            _nested_getattr(model, koop_attr)
            if koop_attr else None
        )

        eig_raw = (
            obj.S_raw
            if isinstance(obj, StrictStableKoopmanOperator)
            else None
        )

        eigs.append(
            0.0 if eig_raw is None
            else torch.sigmoid(eig_raw).abs().max().item()
        )

        # In tiến độ sau mỗi 10% số epoch
        if ep % max(1, epochs // 10) == 0 or ep == epochs - 1:
            print(
                f"Epoch {ep:>4}/{epochs} "
                f"MSE {pred_loss.item():.5f} "
                f"Lyap {(lyap_weight * lyap).item():.5f}"
            )

    # Tạo dự báo cuối cùng trên các cửa sổ huấn luyện
    model.eval()

    with torch.no_grad():
        predictions = model(x_in).cpu().numpy()

    return predictions, np.array(losses), np.array(eigs)

print("ERA5 training function ready.")
print("Epochs:", epochs)
print("Learning rate:", learning_rate)
print("Lyapunov weight:", lyap_weight)
print("Device:", device)


ERA5 training function ready.
Epochs: 4000
Learning rate: 0.0003
Lyapunov weight: 0.1
Device: cuda



## 4. Huấn luyện và lưu checkpoint ERA5

### 4.1. Mục tiêu

Thực nghiệm huấn luyện bốn mô hình với cùng cấu hình dữ liệu và siêu tham số, sau đó so sánh MSE và MAE.

Các mô hình được huấn luyện tuần tự theo thứ tự:

1. LSTM.
2. PatchTST.
3. Autoformer.
4. Informer.

Mỗi mô hình được huấn luyện trong 4000 epoch.

### 4.2. Cơ chế tiếp tục huấn luyện

Trước khi huấn luyện, code kiểm tra checkpoint của từng mô hình theo thứ tự đã định.

Nếu checkpoint tồn tại, notebook đọc lại kết quả và trạng thái ngẫu nhiên đã lưu.

Nếu checkpoint chưa tồn tại, notebook tiếp tục huấn luyện các mô hình còn lại.

Cơ chế này được xây dựng nhằm hỗ trợ tiếp tục thực nghiệm khi quá trình huấn luyện bị gián đoạn.

### 4.3. Cấu hình huấn luyện

| Tham số | Giá trị |
|---|---|
| Patch length | 120 |
| Forecast horizon | 30 |
| d_model | 48 |
| Input channels | 10 |
| Epochs | 4000 |
| Learning rate | 0.0003 |
| Seed | 7 |

Với 400 thời điểm, số cửa sổ huấn luyện là 250.

Kích thước nhãn dự báo là 300 giá trị mỗi mẫu.

### 4.4. Nội dung checkpoint

Mỗi checkpoint lưu:

| Trường | Ý nghĩa |
|---|---|
| `model` | Tên mô hình |
| `model_state_dict` | Trọng số mô hình |
| `pred` | Dự báo sau huấn luyện |
| `truth` | Nhãn thực tế |
| `loss` | Lịch sử hàm mất mát |
| `eig` | Lịch sử chỉ số theo dõi Koopman |
| `mse` | MSE trên dữ liệu huấn luyện |
| `mae` | MAE trên dữ liệu huấn luyện |
| `patch_len` | Độ dài đầu vào |
| `horizon` | Chân trời dự báo |
| `d_model` | Kích thước biểu diễn |
| `input_dim` | Số kênh |
| `epochs` | Số epoch |
| `location_names` | Danh sách 10 địa điểm |
| `rng_torch`, `rng_cuda` | Trạng thái ngẫu nhiên đã lưu |

### 4.5. Quản lý kết quả

Thư mục:

`results/era5_wind_e4000/`

Tên checkpoint có dạng:

`{model}_p120_h30_d48.pt`

Tệp tổng hợp:

`metrics.csv`

Các metric được ghi sau mỗi mô hình hoàn thành.

### 4.6. Kết quả thực nghiệm

Các kết quả đã lưu:

| Mô hình | MSE | MAE |
|---|---:|---:|
| LSTM | 0.003387 | 0.044388 |
| PatchTST | 0.002596 | 0.039073 |
| Autoformer | 0.003411 | 0.044773 |
| Informer | 0.002664 | 0.039703 |

Trong lần huấn luyện này, PatchTST đạt MSE và MAE thấp nhất.

Các chỉ số được tính trên những cửa sổ huấn luyện, do đó chưa thể hiện khả năng tổng quát hóa trên tập Test độc lập.

### 4.7. Bảo toàn kết quả

Bốn checkpoint đã được tạo trong lần chạy trước trên Kaggle.

Không thực hiện huấn luyện lại hoặc ghi đè các checkpoint khi tái cấu trúc notebook.


In [10]:
import gc
import numpy as np
import pandas as pd
import torch

# 1. Experiment configuration
model_names = ["LSTM", "PatchTST", "Autoformer", "Informer"]
metrics_path = results_dir / "metrics.csv"

assert device.type == "cuda", "Enable Kaggle GPU before training"
assert epochs == 4000
assert (patch_len, horizon, d_model) == (120, 30, 48)
assert x_in.shape == (250, 120, 10)
assert x_out.shape == (250, 300)

results_dir.mkdir(parents=True, exist_ok=True)

def checkpoint_path(name):
    return results_dir / (
        f"{name.lower()}_p{patch_len}_h{horizon}_d{d_model}.pt"
    )

def save_metrics(records):
    df = pd.DataFrame(
        records,
        columns=[
            "Model", "PatchLen", "Horizon",
            "d_model", "Epochs", "MSE", "MAE"
        ]
    )

    temp_path = metrics_path.with_suffix(".csv.tmp")
    df.to_csv(temp_path, index=False)
    temp_path.replace(metrics_path)

# 2. Restore completed checkpoints
set_seed(seed)

records = []
completed = 0

for name in model_names:
    path = checkpoint_path(name)

    if not path.is_file():
        break

    ckpt = torch.load(
        path, map_location="cpu", weights_only=True
    )

    assert ckpt["model"] == name
    assert ckpt["epochs"] == epochs
    assert ckpt["patch_len"] == patch_len
    assert ckpt["horizon"] == horizon
    assert ckpt["d_model"] == d_model
    assert ckpt["input_dim"] == input_dim

    records.append({
        "Model": name,
        "PatchLen": patch_len,
        "Horizon": horizon,
        "d_model": d_model,
        "Epochs": epochs,
        "MSE": ckpt["mse"],
        "MAE": ckpt["mae"]
    })

    completed += 1

    # Restore RNG state for sequential training
    torch.set_rng_state(ckpt["rng_torch"])
    torch.cuda.set_rng_state_all(ckpt["rng_cuda"])

# Prevent inconsistent checkpoint order
for name in model_names[completed:]:
    if checkpoint_path(name).exists():
        raise RuntimeError(
            "Non-sequential checkpoints detected. "
            "Check result files before resuming."
        )

save_metrics(records)

print("=== ERA5 WIND SPEED TRAINING ===")
print("GPU:", torch.cuda.get_device_name(0))
print("Patch length:", patch_len)
print("Horizon:", horizon)
print("d_model:", d_model)
print("Epochs:", epochs)
print("Total models:", len(model_names))
print("Completed:", completed)
print("Remaining:", len(model_names) - completed)
print("Results:", results_dir)

# 3. Train remaining models
for idx, name in enumerate(
    model_names[completed:], start=completed + 1
):
    print(f"\n{'=' * 55}")
    print(f"[{idx}/4] Training {name}")
    print(f"{'=' * 55}")

    model = create_model(name)

    is_lstm = name == "LSTM"

    pred, loss, eig = train_era5_model(
        model,
        x_in,
        x_out,
        epochs=epochs,
        lr=learning_rate,
        koop_attr=None if is_lstm else "koop",
        lyap_weight=0.0 if is_lstm else lyap_weight
    )

    # 4. Calculate training metrics
    truth = x_out.cpu().numpy()
    error = pred - truth

    mse_value = float(np.mean(error ** 2))
    mae_value = float(np.mean(np.abs(error)))

    print(f"\n{name} - Final Results")
    print(f"MSE: {mse_value:.6f}")
    print(f"MAE: {mae_value:.6f}")

    # 5. Save checkpoint and training history
    checkpoint = {
        "model": name,
        "model_state_dict": model.cpu().state_dict(),
        "pred": torch.from_numpy(pred.copy()),
        "truth": torch.from_numpy(truth.copy()),
        "loss": torch.from_numpy(loss.copy()),
        "eig": torch.from_numpy(eig.copy()),
        "mse": mse_value,
        "mae": mae_value,
        "patch_len": patch_len,
        "horizon": horizon,
        "d_model": d_model,
        "input_dim": input_dim,
        "epochs": epochs,
        "location_names": list(location_names),
        "rng_torch": torch.get_rng_state(),
        "rng_cuda": torch.cuda.get_rng_state_all()
    }

    path = checkpoint_path(name)
    temp_path = path.with_suffix(".pt.tmp")

    torch.save(checkpoint, temp_path)
    temp_path.replace(path)

    # 6. Update metrics after every model
    records.append({
        "Model": name,
        "PatchLen": patch_len,
        "Horizon": horizon,
        "d_model": d_model,
        "Epochs": epochs,
        "MSE": mse_value,
        "MAE": mae_value
    })

    save_metrics(records)

    print("Checkpoint saved:", path)
    print("Completed models:", len(records), "/ 4")

    del model, pred, loss, eig, checkpoint
    gc.collect()
    torch.cuda.empty_cache()

# 7. Final summary
print("\n=== ERA5 TRAINING COMPLETE ===")
print(pd.read_csv(metrics_path).to_string(index=False))
print("\nResults directory:", results_dir)


=== ERA5 WIND SPEED TRAINING ===
GPU: Tesla T4
Patch length: 120
Horizon: 30
d_model: 48
Epochs: 4000
Total models: 4
Completed: 0
Remaining: 4
Results: /kaggle/working/era5_wind_e4000

[1/4] Training LSTM
Epoch    0/4000 MSE 0.25885 Lyap 0.00000
Epoch  400/4000 MSE 0.02373 Lyap 0.00000
Epoch  800/4000 MSE 0.01523 Lyap 0.00000
Epoch 1200/4000 MSE 0.00904 Lyap 0.00000
Epoch 1600/4000 MSE 0.00723 Lyap 0.00000
Epoch 2000/4000 MSE 0.00605 Lyap 0.00000
Epoch 2400/4000 MSE 0.00522 Lyap 0.00000
Epoch 2800/4000 MSE 0.00466 Lyap 0.00000
Epoch 3200/4000 MSE 0.00413 Lyap 0.00000
Epoch 3600/4000 MSE 0.00378 Lyap 0.00000
Epoch 3999/4000 MSE 0.00339 Lyap 0.00000

LSTM - Final Results
MSE: 0.003387
MAE: 0.044388
Checkpoint saved: /kaggle/working/era5_wind_e4000/lstm_p120_h30_d48.pt
Completed models: 1 / 4

[2/4] Training PatchTST
Epoch    0/4000 MSE 0.34617 Lyap 0.00000
Epoch  400/4000 MSE 0.03170 Lyap 0.00000
Epoch  800/4000 MSE 0.01438 Lyap 0.00000
Epoch 1200/4000 MSE 0.00845 Lyap 0.00000
Epoch 160